# Example Solution: Classification Exercise 2026

Below you will find my thought process tackling this task. Note: I do not know the optimal solution for this problem and I'm sure this solution can be improved further -> either from a landing point of view or classification.

Firstly, I decided to tackle the landing success objective and essentially disregard classification as to me this was more interesting. However, I believe that targetting a high Macro F1 score would correlate with a good landing score.

Before writing any code I brainstormed some useful features based on the physics. I decided to build them up in stages as I was not sure how beneficial each one would be. They all work using some projection of the rocket's state into the future (essentially how a MPC controller would do it too). I did not neccessarily make all the changes in order (i fixed the split etc first). I may have taken some of these values from the simulator, but as one of you found out the simulator wasn't hidden in the browser environment.

#### Change 1: Just using velocity and no acceleration.

There are essentially four states we care about: The position at landing, the contact speed, the contact angle and the angular velocity at contact. So therefore I only used these as my observations/states.

The equations for these are:

$$\hat{y} = y + v_yt$$
$$\hat{v}_y = v_y$$
$$\hat{\theta} = \theta + \omega t$$
$$\hat{\omega} = \omega$$

With no acceleration the velocities are just held constant. I evaluate these at $t = 1, 2, 3\,\text{s}$, and stop the projection at first contact with the pad, so the features become the contact state if the rocket lands within the horizon. Contact is detected using the lowest point of the hull/feet, with each point $(x_j, y_j)$ rotated about the centre of mass $c$:

$$g(\hat{y}, \hat{\theta}) = \hat{y} + \min_j \left[ x_j \sin\hat{\theta} + (y_j - c)\cos\hat{\theta} \right]$$

where $g \le 0$ means the rocket has touched down.

I hypothesised that the more important direction was the y direction so only added this at first.

#### Change 2: Adding in acceleration due to gravity

Using basically the same equations as above, we can add the acceleration due to gravity to the height prediction.

Resulting in:

$$\hat{y} = y + v_yt + \frac{1}{2} g t^2$$
$$\hat{v}_y = v_y + g t$$

where $g = -9.80665\,\text{m/s}^2$ (negative because up is positive). The angle equations are unchanged.

Technically, if the rocket angle is not 0 there would also be some torque due to the weight, however I did not consider this here.

#### Change 3: Adding in the drag

The drag term influences both the vertical position projection and the rotation (as a side wind would rotate the rocket, or if the rocket was off-axis).

This leads to these equations:

$$\hat{y} = y + v_yt + \frac{1}{2} a_y t^2$$
$$\hat{v}_y = v_y + a_y t$$
$$\hat{\theta} = \theta + \omega t + \frac{1}{2} \alpha t^2$$
$$\hat{\omega} = \omega + \alpha t$$

where:
$$a_y = \frac{F_y}{m} + g, \qquad \alpha = \frac{\tau}{I}$$

The drag force opposes the velocity *relative to the wind* $\mathbf{w}$:
$$\mathbf{v}_{rel} = \mathbf{v} - \mathbf{w}, \qquad \mathbf{F} = -\frac{1}{2}\rho C_d A\, |\mathbf{v}_{rel}|\, \mathbf{v}_{rel}$$

The drag coefficient depends on the angle of attack $\beta$ (the angle between the rocket axis and $\mathbf{v}_{rel}$), since the rocket has far more drag side-on than end-on:
$$C_d = 0.65 + 15 \sin^2\beta$$

The drag acts at the centre of pressure $c_p$ (37 m up the body), which is above the centre of mass $c$, so it produces a torque:
$$\tau = (c_p - c)\left(-F_y \sin\theta - F_x \cos\theta\right)$$

Since drag depends on the velocity, $a_y$ and $\alpha$ change over time, so instead of using one big $t$ I apply these equations in small steps of $\Delta t = 0.25\,\text{s}$ and recalculate the drag at each step.

#### Change 4: Adding in thrust

The observation also contains the current thrust $T$ and gimbal angle $\delta$. Holding these fixed over the horizon (i.e. assuming the controller keeps doing what it's doing), they add to the force and torque:

$$F_x \mathrel{+}= -T \sin(\theta + \delta), \qquad F_y \mathrel{+}= T \cos(\theta + \delta)$$
$$\tau \mathrel{+}= (c_e - c)\, T \sin\delta$$

where $c_e$ is the engine position (1 m up the body). Thrust is set to 0 if the rocket is out of fuel. I guess if you think about it, this doesn't really make sense as a basis function as this means the reference point changes depending on the previous action and therefore doesn't really make for stable basis functions.

#### Change 5: Adding the horizontal direction - this I did after the exercise class (you can find the result on the leaderboard in position 5)

Going back to Change 3 (without thrust), I also project the horizontal position and velocity with the same step-wise update:

$$\hat{x} = x + v_xt + \frac{1}{2} a_x t^2$$
$$\hat{v}_x = v_x + a_x t$$

where $a_x = F_x / m$ comes only from drag (no gravity in $x$). The pad centre is at $x = 0$, so $\hat{x}$ is directly how far off the pad the rocket would land.

#### Fix 1: Normalisation

The first fix is in the scaffold, which I think everyone saw and changed. The normalisation was originally applied regardless of whether the model was in training or prediction. This is wrong for two reasons:
1. It would fit the normalisation on the validation set, but use the weights trained with the training normalisation. Therefore it would introduce a shift that would likely make the model perform a lot worse.
2. In predict action it is only fed one observation, so when you try to compute the standard deviation you get 0. Therefore you introduce a divide by 0 error.

#### Fix 2: Splits

This one everyone applied too. The original split was random but with this data the previous state is directly related to the next one. Therefore if this previous state appears in your test data but you have the next state in your training data you essentially have already learned about this position. This is temporal leakage. In order to fix it you can split each run by time or you can just split by run. I favour the latter so you can learn about what actions the model takes near the platform.

#### Fix 3: Class-weighting or class sampling

I applied class weighting as I saw that there was a severe imbalance in the data. A model could get a decent accuracy and therefore loss
